# Lab 2E: RAG Pipeline in Python

**Time**: ~30 min  
**Environment**: Jupyter kernel in VS Code  

In this exercise you will build a Retrieval-Augmented Generation (RAG) pipeline using Cosmos DB vector search and Azure OpenAI.

The lab follows the same structure as the C# version. Run each cell in order to complete the steps.

In [ ]:
%pip install azure-cosmos azure-identity openai python-dotenv numpy --quiet

## Step 0: Initialize Connection

Set up the Cosmos DB and Azure OpenAI client connections.

In [ ]:
from dotenv import load_dotenv
load_dotenv()

import os

ENDPOINT = os.environ.get("COSMOS_ENDPOINT")
FOUNDRY_ENDPOINT = os.environ.get("FOUNDRY_ENDPOINT")
EMBEDDINGS_ENDPOINT = os.environ.get("EMBEDDINGS_ENDPOINT")
EMBEDDINGS_KEY = os.environ.get("EMBEDDINGS_KEY")
DB_NAME = "WorkshopData"
CONT_NAME = "Docs"
COMPLETIONS_MODEL = os.environ.get("COMPLETIONS_MODEL", "phi-4-mini-instruct")
EMBEDDINGS_MODEL = os.environ.get("EMBEDDINGS_MODEL", "text-embedding-3-small")

for var in ["COSMOS_ENDPOINT", "FOUNDRY_ENDPOINT", "EMBEDDINGS_ENDPOINT", "EMBEDDINGS_KEY"]:
    if not os.environ.get(var):
        raise RuntimeError(f"{var} environment variable is required.")

print(f"Cosmos Endpoint:     {ENDPOINT}")
print(f"Foundry Endpoint:    {FOUNDRY_ENDPOINT}")
print(f"Embeddings Endpoint: {EMBEDDINGS_ENDPOINT}")
print(f"Database:            {DB_NAME}")
print(f"Container:           {CONT_NAME}")
print(f"Completions Model:   {COMPLETIONS_MODEL}")
print(f"Embeddings Model:    {EMBEDDINGS_MODEL}")

In [ ]:
from azure.cosmos import CosmosClient, PartitionKey
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from openai import OpenAI

# Cosmos DB client (Entra ID auth)
cred = DefaultAzureCredential()
cosmos_client = CosmosClient(url=ENDPOINT, credential=cred)
db = cosmos_client.get_database_client(DB_NAME)
container = db.get_container_client(CONT_NAME)
print(f"Connected to: {ENDPOINT}/{DB_NAME}/{CONT_NAME}")

# Chat completions: Foundry endpoint, Entra ID auth.
token_provider = get_bearer_token_provider(cred, "https://ai.azure.com/.default")
foundry_client = OpenAI(
    base_url=f"{FOUNDRY_ENDPOINT.rstrip('/')}/openai/v1/",
    api_key=token_provider,
)

# Embeddings: separate Azure OpenAI resource, API key auth
# (the v1 embeddings surface does not yet support Entra ID).
embeddings_client = OpenAI(
    base_url=f"{EMBEDDINGS_ENDPOINT.rstrip('/')}/openai/v1/",
    api_key=EMBEDDINGS_KEY,
)
print("Foundry chat client + embeddings client initialized")

## Step 1: Text Chunking and Seed Documents (Prebuilt)

Loads sample documents and chunks them into 512-character segments.

In [ ]:
def chunk_text(text: str, chunk_size: int = 512) -> list[str]:
    sentences = text.split(". ")
    chunks = []
    current_chunk = []
    current_size = 0
    
    for sentence in sentences:
        if current_size + len(sentence) > chunk_size and current_chunk:
            chunks.append(". ".join(current_chunk) + ".")
            current_chunk = [sentence]
            current_size = len(sentence)
        else:
            current_chunk.append(sentence)
            current_size += len(sentence)
    
    if current_chunk:
        chunks.append(". ".join(current_chunk) + ".")
    
    return chunks


sample_docs = [
    {"id": "doc1", "title": "Cosmos DB Overview", "content": "Azure Cosmos DB is a globally distributed, multi-model database service. It supports multiple API modes including SQL, MongoDB, Cassandra, Table, and Gremlin. Cosmos DB provides five consistency levels: Strong, Bounded Staleness, Session, Consistent Prefix, and Eventual."},
    {"id": "doc2", "title": "Vector Search", "content": "Azure Cosmos DB supports vector indexing for semantic similarity search queries. Vector search enables finding semantically similar items by comparing embeddings. The VectorDistance function calculates similarity scores between vectors."},
    {"id": "doc3", "title": "Data Modeling", "content": "Effective data modeling in Cosmos DB involves choosing the right partition key. Composite partition keys can help distribute load across partitions. Denormalization and fan-out patterns can optimize read performance."}
]

for doc in sample_docs:
    chunks = chunk_text(doc["content"])
    print(f"  Chunked '{doc['title']}': {len(chunks)} chunks")

print(f"Total: {len(sample_docs)} documents loaded")

## Step 2: Embed and Store Chunks (STUDENT EXERCISE)

Generate embeddings for each chunk and store them in the Cosmos DB `Docs` container with vector index.

**Expected output**: Chunks stored with their embeddings.

**Hint**: Use `embeddings_client.embeddings.create(input=text, model=model)` to get embeddings.

In [ ]:
def get_embedding(text: str) -> list[float]:
    resp = embeddings_client.embeddings.create(input=text, model=EMBEDDINGS_MODEL)
    return resp.data[0].embedding


for doc in sample_docs:
    chunks = chunk_text(doc["content"])

    for i, chunk in enumerate(chunks):
        embedding = get_embedding(chunk)

        stored_doc = {
            "id": f"{doc['id']}_chunk_{i}",
            "title": doc["title"],
            "text": chunk,
            "embedding": embedding,
            "source": doc["id"],
            "partitionKey": "rag"
        }

        try:
            container.upsert_item(body=stored_doc, partition_key="rag")
            ru = float(container.client_connection.last_response_headers["x-ms-request-charge"])
            print(f"  Stored chunk {i+1} of {doc['title']} (chunks={len(chunks)})")
            print(f"  RU charged: {ru}")
        except Exception as ex:
            print(f"  Error storing chunk: {ex}")

print("Chunk embedding and storage complete.")

## Step 3: RAG Retrieval (STUDENT EXERCISE)

Query the vector index to retrieve the most relevant document chunks for a given search query.

**Expected output**: 3 most relevant document chunks with scores.

In [ ]:
def retrieve_relevant(text_query: str, top_k: int = 3) -> list[dict]:
    query_embedding = get_embedding(text_query)

    vector_query = """
    SELECT TOP @topk c.text, c.title, VectorDistance(c.embedding, @emb) AS score
    FROM c
    WHERE c.partitionKey = 'rag'
    ORDER BY VectorDistance(c.embedding, @emb)
    """

    results = list(container.query_items(
        query=vector_query,
        parameters=[
            {"name": "@topk", "value": top_k},
            {"name": "@emb", "value": query_embedding}
        ],
        partition_key="rag"
    ))

    return results


test_query = "What is vector search in Azure Cosmos DB?"
print(f"Retrieving for: '{test_query}'\n")

results = retrieve_relevant(test_query, 3)
print(f"Found {len(results)} results:\n")
for result in results:
    text = result.get("text", "")
    print(f"  Title: {result.get('title')}")
    print(f"  Score: {result.get('score')}")
    print(f"  Text: {text[:100]}...\n")

## Step 4: RAG Generation (STUDENT EXERCISE)

Combine the retrieved context with a chat model to generate a response.

**Expected output**: A generated answer based on the retrieved context.

In [ ]:
def generate_response(question: str) -> str:
    results = retrieve_relevant(question, 3)
    context = "\n\n".join(r.get("text", "") for r in results)

    system_prompt = f"You are a helpful assistant. Answer the user's question based on the following context:\n\n<context>{context}</context>"

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question}
    ]

    completion = foundry_client.chat.completions.create(
        model=COMPLETIONS_MODEL,
        messages=messages,
        temperature=0.7,
        max_tokens=500
    )

    return completion.choices[0].message.content


print("=== Testing RAG Pipeline\n")

test_question = "What is vector search in Azure Cosmos DB?"
print(f"Question: {test_question}")

answer = generate_response(test_question)
print(f"\nAnswer:\n{answer}")

print("\n=== Lab Complete ===")
print("You have completed the RAG Pipeline exercise in Python. You:")
print("- Chunked documents into segments")
print("- Generated embeddings for each chunk using Azure OpenAI")
print("- Stored chunks with embeddings in Cosmos DB")
print("- Retrieved relevant chunks using vector search")
print("- Combined retrieved context with chat completions for RAG")